In [1]:
from random import seed, choice, shuffle
import numpy as np
import logging
import heapq
from collections.abc import Iterator
from collections.abc import Callable

logging.getLogger().setLevel(logging.INFO)

In [2]:
class UPQ:
    _data: list[tuple[object, object]]
    _priority: dict

    def __init__(self, data=tuple()) -> None:
        """Create a new UPQ, possibly initialized with data."""
        self._data = list()
        self._priority = dict()
        for i in data:
            self.push_update(*i)

    def __getitem__(self, item) -> object:
        """Return the priority of an item."""
        return self._priority[item]

    def __len__(self) -> int:
        """Return the number of items in the queue."""
        return len(self._priority)

    def __bool__(self) -> bool:
        """Return True if the queue is not empty."""
        return bool(self._priority)

    def __delitem__(self, item) -> None:
        """Delete an item from the queue."""
        del self._priority[item]

    def __contains__(self, item) -> bool:
        """Check if item is in the queue."""
        return item in self._priority

    def __iter__(self) -> Iterator[tuple[object, object]]:
        """Iterate in priority order without modifying the queue."""
        return iter(sorted(self._priority.items(), key=lambda k: k[1]))

    def get(self, item, default=None) -> object:
        """Return the priority of an item, or default if not present."""
        return self._priority.get(item, default)

    def push_update(self, item, priority) -> None:
        """Push a new item with the given priority or update its priority."""
        self._priority[item] = priority
        heapq.heappush(self._data, (priority, item))

    def pop(self) -> tuple[object, object]:
        """Pop the item with the lowest priority. Fails if empty."""
        p, i = heapq.heappop(self._data)
        while p != self._priority.get(i):
            p, i = heapq.heappop(self._data)
        del self._priority[i]
        return (i, p)

    def pop_safe(self, default=None) -> tuple[object, object] | object:
        """Pop the item with the lowest priority. Return `default` if the queue is empty."""
        while self._data:
            p, i = heapq.heappop(self._data)
            if p == self._priority.get(i):
                del self._priority[i]
                return (i, p)
        else:
            return default

In [3]:
class State:
    def __init__(self, data: np.ndarray):
        self._data = data.copy()
        self._data.flags.writeable = False

    def __hash__(self):
        return hash(bytes(self._data))

    def __eq__(self, other):
        return bytes(self._data) == bytes(other._data)

    def __lt__(self, other):
        return bytes(self._data) < bytes(other._data)

    def __str__(self):
        return str(self._data)

    def __repr__(self):
        return repr(self._data)

    @property
    def data(self):
        return self._data

    def copy_data(self):
        return self._data.copy()

In [4]:
class Piece:
    def __init__(self, label, occupied_squares):
        self._label = label
        self._occupied_squares = occupied_squares

    def label(self):
        return self._label

    def moves(self):
        return [np.array(_) for _ in [(-1, 0), (0, +1), (+1, 0), (0, -1)]]

    def find_empty_space(self, board: np.ndarray):
        t = np.where(board == -1)
        return np.array([t[0], t[1]])

    def is_valid(self, board: np.ndarray, action):
        for occupied_square in self._occupied_squares:
            move = occupied_square + action
            # for free_space in self.find_empty_space(board):
            #     check_borders = all(0 <= (free_space + action)[i] < board.shape[i] for i in [0, 1])
            #     if not check_borders:
            #         return False
            if move[0] < 0 or move[0] >= board.shape[0] or move[1] < 0 or move[1] >= board.shape[1]:
                return False
            if not board[move[0], move[1]] in [-1, self.label()]:
                return False
        return True

    def possible_actions(self, state: State):
        moves = self.moves()
        return (m for m in moves if self.is_valid(state._data, m))

    def piece_moving(self, action):
        new_squares = []
        for occupied_square in self._occupied_squares:
            move = occupied_square + action
            new_squares.append(move)
        return Piece(self._label, new_squares)


In [5]:
seed(42)
GOAL = State(np.array([[-1, -1, 1, 2], [3, 4, 5, 6], [7, 0, 0, 8], [9, 0, 0, 10]]))
logging.info(f"Goal: \n{GOAL}")

INFO:root:Goal: 
[[-1 -1  1  2]
 [ 3  4  5  6]
 [ 7  0  0  8]
 [ 9  0  0 10]]


In [6]:
class Board:
    def __init__(self, pieces, squares):
        self._pieces = pieces
        self._squares = squares

    def change_setup(self):
        boards_list = []
        new_board = Board(self._pieces, self._squares)
        squares = self._squares
        copy_pieces = self._pieces
        for piece in self._pieces:
            possible_actions = list(piece.possible_actions(squares))
            if len(possible_actions) > 0:
                choose_action = choice(possible_actions)
                next_piece = piece.piece_moving(choose_action)
                new_board = self.make_move(piece, next_piece, copy_pieces, squares)
                squares = new_board._squares
                copy_pieces = new_board._pieces
                boards_list.append(new_board)
            else:
                continue
        return new_board

    def make_move(self, old_piece, new_piece, list_pieces, update_squares):
        assert old_piece.label() == new_piece.label()
        label = new_piece.label()

        new_pieces = list(list_pieces)
        new_pieces[label] = new_piece
        new_squares = update_squares.copy_data()
        for empty_place in old_piece._occupied_squares:
            new_squares[empty_place[0], empty_place[1]] = -1

        for fill_place in new_piece._occupied_squares:
            new_squares[fill_place[0], fill_place[1]] = label

        return Board(new_pieces, State(new_squares))

    def search(self, initial_state: State, goal_test: Callable, parent_state: dict,
               state_cost: dict, priority_function: Callable, unit_cost: Callable):
        frontier = UPQ()
        parent_state.clear()
        state_cost.clear()

        state = initial_state
        parent_state[state] = None
        state_cost[state] = 0
        while state is not None and goal_test(state):
            squares = self._squares
            copy_pieces = self._pieces
            for piece in self._pieces:
                for a in piece.possible_actions(state):
                    next_piece = piece.piece_moving(a)
                    new_board = self.make_move(piece, next_piece, copy_pieces, squares)
                    squares = new_board._squares
                    copy_pieces = new_board._pieces
                    cost = unit_cost(a)
                    if squares not in state_cost and squares not in frontier:
                        parent_state[squares] = state
                        state_cost[squares] = state_cost[state] + cost
                        frontier.push_update(squares, priority_function(squares))
                        logging.debug(f"Added new node to frontier (cost={state_cost[squares]})")
                    elif squares in frontier and state_cost[squares] > state_cost[state] + cost:
                        old_cost = state_cost[squares]
                        parent_state[squares] = state
                        state_cost[squares] = state_cost[state] + cost
                        frontier.push_update(squares, priority_function(squares))
                        logging.debug(
                            f"Updated node cost in frontier: {old_cost} -> {state_cost[squares]}"
                        )
                if frontier:
                    state, _ = frontier.pop()
                else:
                    state = None
         path = list()
        s = state
        while s:
            path.append(s.copy_data())
            s = parent_state[s]

        logging.info(f"Found a solution in {len(path):,} steps; visited {len(state_cost):,} states")
        return list(reversed(path))

pieces = []

init_setup = GOAL.copy_data()
for label_to_find in range(11):
    occupied_squares = [np.array([val_row, val_col])
                        for (val_row, i) in enumerate(init_setup)
                        for (val_col, j) in enumerate(i)
                        if j == label_to_find]
    pieces.append(Piece(label_to_find, occupied_squares))

init_board = Board(pieces, GOAL)
for r in range(5_000):
    init_board = init_board.change_setup()

array([[ 9,  0,  0,  7],
       [-1,  0,  0,  3],
       [ 1, 10,  5,  6],
       [ 4, -1,  2,  8]])